In [12]:
import os
os.environ["HF_HOME"] = "/home/jovyan/.cache/huggingface"

import torch
import polars as pl
from pathlib import Path
import time
import numpy as np
from evo2 import Evo2

# configuration
DATA_ROOT = Path.home() / "vambersky_t/Data"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"


LAYER = "blocks.28.mlp.l3"
SEQ_LEN = 10_000
BIN_SIZE = 50
N_BINS = SEQ_LEN // BIN_SIZE
EMB_DIM = 4096

TFS_TO_PROCESS = Path("tfs_to_process.txt").read_text().splitlines()
CHECKPOINT_EVERY = 500

assert DATA_ROOT.exists(),   f"Data root missing: {DATA_ROOT}"
assert WINDOWS_DIR.exists(), f"Windows dir missing: {WINDOWS_DIR}"
assert N_BINS == 200

print(f"Bins: {N_BINS} × {BIN_SIZE} bp = {SEQ_LEN} bp")
print(f"Per-peak tensor: ({N_BINS}, {EMB_DIM}) float16")
print(f"Checkpoint interval: every {CHECKPOINT_EVERY} peaks")

# load model
print(f"Loading model...")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = Evo2("evo2_7b")
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Bins: 200 × 50 bp = 10000 bp
Per-peak tensor: (200, 4096) float16
Checkpoint interval: every 500 peaks
Loading model...


[04/11/26 16:24:47] INFO     httpx - INFO - HTTP Request: GET                                       ]8;id=14785024;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14785025;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/arcinstitute/evo2_7b/revision/main                  
                             "HTTP/1.1 200 OK"                                                                     

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 22580.37it/s]

Found complete file in repo: evo2_7b.pt


                    INFO     StripedHyena - INFO - Initializing StripedHyena with config:              ]8;id=14785030;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=14785031;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#627\627]8;;\
                             {'model_name': 'shc-evo2-7b-8k-2T-v2', 'vocab_size': 512, 'hidden_size':              
                             4096, 'num_filters': 4096, 'hcl_layer_idxs': [2, 6, 9, 13, 16, 20, 23,                
                             27, 30], 'hcm_layer_idxs': [1, 5, 8, 12, 15, 19, 22, 26, 29],                         
                             'hcs_layer_idxs': [0, 4, 7, 11, 14, 18, 21, 25, 28], 'attn_layer_idxs':               
                             [3, 10, 17, 24, 31], 'hcm_filter_length': 128, 'hcl_filter_groups': 4096,             
                             'hcm_filter_groups': 256, 'hcs_filter_groups': 256, 'hcs_filter_length':              
                             7, 'num_layers': 32, 'short_filter_length': 3, 'num_attention_heads': 32,             
                             'short_filter_bias': False, 'mlp_init_method': 'torch.nn.init.zeros_',                
                             'mlp_output_init_method': 'torch.nn.init.zeros_', 'eps': 1e-06,                       
                             'state_size': 16, 'rotary_emb_base': 10000, 'rotary_emb_scaling_factor':              
                             128, 'use_interpolated_rotary_pos_emb': True,                                         
                             'make_vocab_size_divisible_by': 8, 'inner_size_multiple_of': 16,                      
                             'inner_mlp_size': 11264, 'log_intermediate_values': False, 'proj_groups':             
                             1, 'hyena_filter_groups': 1, 'column_split_hyena': False, 'column_split':             
                             True, 'interleave': True, 'evo2_style_activations': True,                             
                             'model_parallel_size': 1, 'pipe_parallel_size': 1, 'tie_embeddings':                  
                             True, 'mha_out_proj_bias': True, 'hyena_out_proj_bias': True,                         
                             'hyena_flip_x1x2': False, 'qkv_proj_bias': False,                                     
                             'use_fp8_input_projections': False, 'max_seqlen': 1048576,                            
                             'max_batch_size': 1, 'final_norm': True, 'use_flash_attn': True,                      
                             'use_flash_rmsnorm': False, 'use_flash_depthwise': False, 'use_flashfft':             
                             False, 'use_laughing_hyena': False, 'inference_mode': True,                           
                             'tokenizer_type': 'CharLevelTokenizer', 'prefill_style': 'fft',                       
                             'mlp_activation': 'gelu', 'print_activations': False, 'Loader': <class                
                             'yaml.loader.FullLoader'>}                                                            

...

                    INFO     StripedHyena - INFO - Initialized model                                   ]8;id=14785432;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=14785433;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#691\691]8;;\

                    INFO     vortex.model.utils - INFO - Loading                                        ]8;id=14785438;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/utils.py\utils.py]8;;\:]8;id=14785439;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/utils.py#92\92]8;;\
                             /home/jovyan/.cache/huggingface/hub/models--arcinstitute--evo2_7b/snapshot            
                             s/bda0089f92582d5baabf0f22d9fc85f3588f6b58/evo2_7b.pt                                 

Extra keys in state_dict: {'blocks.11.projections._extra_state', 'blocks.8.projections._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.2.mixer.mixer.filter.t', 'blocks.29.projections._extra_state', 'blocks.17.mixer.dense._extra_state', 'unembed.weight', 'blocks.20.projections._extra_state', 'blocks.6.projections._extra_state', 'blocks.22.projections._extra_state', 'blocks.12.projections._extra_state', 'blocks.21.projections._extra_state', 'blocks.23.mixer.mixer.filter.t', 'blocks.26.projections._extra_state', 'blocks.5.projections._extra_state', 'blocks.25.projections._extra_state', 'blocks.7.projections._extra_state', 'blocks.31.mixer.dense._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.16.mixer.mixer.filter.t', 'blocks.10.mixer.dense._extra_state', 'blocks.3.mixer.attn._extra_state', 'blocks.23.projections._extra_state', 'blocks.9.projections._extra_state', 'blocks.30.projections._extra_state', 'blocks.14.projections._extra_state', 'blocks.2.projections._extra_s

[04/11/26 16:24:49] INFO     StripedHyena - INFO - Adjusting Wqkv for column split (permuting rows)    ]8;id=14785444;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=14785445;file:///home/jovyan/envs/evo2/lib/python3.12/site-packages/vortex/model/model.py#976\976]8;;\

Model loaded in 5.0s
VRAM allocated: 13.27 GB


In [13]:
# get the embeddings - mean of every 50, non overlapping (probrat s Vojtou - rozlišení, porovnatelnost)

def embed_sequence(sequence: str) -> np.ndarray:
    """
    Forward-pass one 10 kb sequence through Evo 2, extract embeddings
    from LAYER, bin into (200, 4096) via reshape + mean, return as
    float16 numpy array.

    Uses module-level `model` and `LAYER`.
    """
    input_ids = torch.tensor(
        model.tokenizer.tokenize(sequence),
        dtype=torch.int,
    ).unsqueeze(0).to("cuda:0")

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[LAYER],
        )

    emb = embeddings[LAYER][0]                 # (10000, 4096)

    # reshape + mean - as i did in the test, this works.
    binned = emb.reshape(N_BINS, BIN_SIZE, -1).mean(dim=1)

    return binned.cpu().to(torch.float16).numpy()

# test, delete
test_df = pl.read_csv(
    next((WINDOWS_DIR / "CTCF").glob("*.full_peaks.csv.gz")),
    n_rows=1,
)
test_emb = embed_sequence(test_df["sequence"][0])
print(test_emb.shape, test_emb.dtype) 

(200, 4096) float16
